# Notebook 07 — Loss Functions, Regularization & Ablation Study (Member 3)

This notebook documents the experiments investigating how different **loss balancing strategies**, **regularization techniques**, and **data augmentations** affect the multitask indoor positioning model.

## Experiments

| Run | Loss | Dropout | Augmentation |
|-----|------|---------|-------------|
| A | Basic combined (sum) | 0.2 | None |
| B | Uncertainty weighting | 0.2 | None |
| C | Basic combined | 0.0 | None |
| D | Basic combined | 0.3 | None |
| E | Basic combined | 0.2 | RSSI noise |
| F | Basic combined | 0.2 | AP dropout |
| G | Uncertainty weighting | 0.2 | RSSI + AP dropout |
| WD_1e5 | Basic combined | 0.2 | None (wd=1e-5) |
| WD_1e4 | Basic combined | 0.2 | None (wd=1e-4) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, Image

REPO = Path('..')
METRICS = REPO / 'results' / 'metrics'
FIGURES = REPO / 'results' / 'figures'

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

## 1. Load Ablation Results

In [ ]:
results = pd.read_csv(METRICS / 'ablation_results_member3.csv')

# Display clean summary
summary = results[[
    'experiment', 'loss_type', 'dropout', 'weight_decay', 'augmentation',
    'best_epoch', 'training_time_s',
    'test_building_acc', 'test_floor_acc', 'test_mean_error_m',
    'test_median_error_m', 'test_p90_error_m', 'test_score'
]].copy()

summary['test_building_acc'] = (summary['test_building_acc'] * 100).round(2)
summary['test_floor_acc'] = (summary['test_floor_acc'] * 100).round(2)
summary['test_mean_error_m'] = summary['test_mean_error_m'].round(2)
summary['test_median_error_m'] = summary['test_median_error_m'].round(2)
summary['test_p90_error_m'] = summary['test_p90_error_m'].round(2)
summary['test_score'] = summary['test_score'].round(3)

display(summary)

## 2. Uncertainty-Weighted Loss — Mathematical Foundation

The multitask model optimises three losses simultaneously:
- **Building**: Cross-entropy (classification)
- **Floor**: Cross-entropy (classification)  
- **Coordinates**: MSE (regression)

The basic approach sums them equally: $\mathcal{L}_{total} = \mathcal{L}_{bld} + \mathcal{L}_{flr} + \mathcal{L}_{coord}$

**Problem**: These losses operate on different scales. The regression loss on standardised coordinates can be orders of magnitude different from classification cross-entropy.

**Solution (Kendall et al., 2018)**: Learn task-specific log-variance parameters $s_i$:

$$\mathcal{L}_{total} = \sum_i \left[ \exp(-s_i) \cdot \mathcal{L}_i + c_i \cdot s_i \right]$$

where $c_i = 1.0$ for classification, $c_i = 0.5$ for regression.

In [ ]:
# Show the UncertaintyLoss implementation
from src.models.loss import UncertaintyLoss
import inspect
print(inspect.getsource(UncertaintyLoss))

## 3. Augmentation Strategies

In [ ]:
from src.augmentations import RSSINoise, APDropout
import torch

# Demonstrate augmentations on a sample scan
torch.manual_seed(42)
# Simulate a scan: 10 APs, most undetected (0.0), a few detected
scan = torch.tensor([[0.0, 0.3, 0.0, 0.7, 0.0, 0.5, 0.0, 0.0, 0.2, 0.0]])

print('Original scan:', scan.numpy().round(3))
print('RSSI Noise:   ', RSSINoise(std=0.02)(scan).numpy().round(3))
print('AP Dropout:   ', APDropout(drop_rate=0.3)(scan).numpy().round(3))
print()
print('Note: sentinel values (0.0 = undetected AP) are NEVER modified.')

## 4. Results Visualization

In [ ]:
# Bar comparison across all metrics
display(Image(filename=str(FIGURES / 'ablation_bar_comparison.png')))

In [ ]:
# EvAAL competition score comparison (lower is better)
display(Image(filename=str(FIGURES / 'ablation_evaal_scores.png')))

In [ ]:
# Training curves
display(Image(filename=str(FIGURES / 'ablation_training_curves.png')))

In [ ]:
# Uncertainty weight evolution (Experiments B and G)
display(Image(filename=str(FIGURES / 'ablation_uncertainty_weights.png')))

## 5. Analysis & Key Findings

### Best Experiment: D (Dropout 0.3)
- **EvAAL Score: 9.359** (best among all 9 experiments)
- Floor accuracy improved to 91.81% (+0.81% over baseline)
- Mean coordinate error reduced to 8.85m (−0.10m)

### Dropout Analysis (C vs A vs D)
| Dropout | EvAAL | Floor Acc | Mean Error |
|---------|-------|-----------|------------|
| 0.0 (C) | 10.117 | 91.00% | 9.49m |
| 0.2 (A) | 9.496 | 91.00% | 8.95m |
| **0.3 (D)** | **9.359** | **91.81%** | **8.85m** |

→ Removing dropout caused clear overfitting (+0.62 EvAAL). Increasing to 0.3 improved all metrics.

### Loss Weighting (B vs A)
Uncertainty weighting achieved comparable classification but worse coordinate error (9.19m vs 8.95m). The adaptive weights converge to down-weighting the coordinate task, suggesting the equal-weight baseline is already well-balanced for this architecture.

### RSSI Noise (E) — Building Robustness
RSSI noise improved building accuracy from 99.73% to 99.91% and achieved the second-best EvAAL (9.423). This confirms that signal noise augmentation helps generalise to the 9 unseen phone models in the test set.

### AP Dropout (F) — Mixed Results
AP dropout matched the building accuracy improvement (99.91%) but degraded floor accuracy to 90.19%. Floor discrimination relies on subtle multi-AP spatial patterns that are disrupted by aggressive AP masking.

### Weight Decay — Counterproductive
Both weight decay values degraded performance, with λ=1e-4 producing the worst EvAAL (10.754). The coordinate regression head requires sufficient weight capacity that L2 regularisation restricts.

In [ ]:
# Delta analysis relative to baseline A
baseline = results[results['experiment'] == 'A'].iloc[0]

deltas = results[['experiment', 'description']].copy()
deltas['Δ Bld Acc (pp)'] = ((results['test_building_acc'] - baseline['test_building_acc']) * 100).round(2)
deltas['Δ Flr Acc (pp)'] = ((results['test_floor_acc'] - baseline['test_floor_acc']) * 100).round(2)
deltas['Δ Mean Err (m)'] = (results['test_mean_error_m'] - baseline['test_mean_error_m']).round(2)
deltas['Δ EvAAL'] = (results['test_score'] - baseline['test_score']).round(3)

display(deltas)

## 6. Recommendation

For the final model configuration, **Experiment D** (dropout=0.3 with basic combined loss) is recommended as the primary training setting. RSSI noise augmentation (Experiment E) is a secondary recommendation for deployment scenarios requiring device-agnostic robustness.

---
*Member 3 deliverables complete: loss implementation, regularization code, augmentation code, ablation results, comparison graphs, and mathematical justification.*